In [75]:
import numpy as np 
from scipy import sparse 
from matplotlib import pyplot as plt
from matplotlib import animation

In [76]:
def animate_heat_2d(U, X, Y, title, filename, interval=50):

    # create figure and axes
    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title(title)

    # update function
    def update(t):
        ax.clear()
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
        ax.set_zlim([0, 1])
        ax.plot_surface(X, Y, U[t], cmap="magma")
        
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("z")
        ax.set_title(title)

    # create and save animation
    ani = animation.FuncAnimation(fig, update, U.shape[0], interval=interval)
    plt.close(fig)
    ani.save(filename)

In [77]:
def explicit_heat_solver_2d(nu, u0, x_span, y_span, tf, nx, ny, nt):

    # define space and time domains
    x_vals, dx = np.linspace(*x_span, nx+1, retstep=True)
    y_vals, dy = np.linspace(*y_span, ny+1, retstep=True)
    t_vals, dt = np.linspace(0, tf, nt+1, retstep=True)
    X, Y = np.meshgrid(x_vals, y_vals)

    # compute CFL values
    lam_x = nu * dt / dx**2
    lam_y = nu * dt / dy**2

    # compute the initial state
    U_init = np.array(
        [[u0(X[i, j], Y[i, j]) for i in range(X.shape[0])] for j in range(X.shape[1])]
    )
    
    # build stiffness matrix
    T = sparse.diags(
        diagonals=[lam_x, 1. - 2*lam_x - 2*lam_y, lam_x],
        offsets=[-1, 0, 1],
        shape=(nx-1, nx-1),
    )
    A = sparse.block_diag([T] * (ny-1))
    A.setdiag(lam_y, -(nx-1))
    A.setdiag(lam_y, nx-1)

    # initialize solution matrix -- homogenous Dirichlet BCs
    U = np.zeros((nt+1, nx+1, ny+1))
    sl = np.s_[1:-1]                    # interior slice 
    U[0, sl, sl] = U_init[sl, sl]       # set initial state on interior

    # solver loop 
    for t in range(1, nt+1):
        # this only works for homogenous Dirichlet BCs
        # if non-homoegenous, there should be an additive term on the right side
        U[t, sl, sl] = (A @ U[t-1, sl, sl].flatten()).reshape((nx-1, ny-1))

    return U, X, Y, t_vals

In [78]:
# model parameters 
nu = 1.             # diffusion coefficient
x0, xf = 0, 1       # space domain (x)
y0, yf = 0, 1       # space domain (y)
tf = 0.25           # time domain
def u0(x, y):       # initial state
    return (
        np.sin(np.pi * (x - x0) / (xf - x0)) * 
        np.sin(np.pi * (y - y0) / (yf - y0))
    )

# # initial state (Gaussian bump)
# x_mid = (xf - x0) / 2.
# y_mid = (yf - y0) / 2.
# u0 = lambda x, y : np.exp((x - x_mid)**2 + (y - y_mid)**2)

# system parameters 
nx = 10 
ny = 10
nt = 100 

# test solver 
U, X, Y, t_vals = explicit_heat_solver_2d(nu, u0, (x0, xf), (y0, yf), tf, nx, ny, nt)

animate_heat_2d(
    U, X, Y,
    "Explicit Solver for 2D Heat Equation",
    "explicit_2d.mp4",
    interval=50
)

<video src="explicit_2d.mp4" controls> 

In [79]:
# system parameters 
nx = 20 
ny = 20
nt = 100 

# test solver with broken CFL condition
U, X, Y, t_vals = explicit_heat_solver_2d(nu, u0, (x0, xf), (y0, yf), tf, nx, ny, nt)

animate_heat_2d(
    U, X, Y,
    "Explicit Solver for 2D Heat Equation, Unstable",
    "explicit_2d_unstable.mp4",
    interval=50
)

<video src="explicit_2d_unstable.mp4" controls> 